<a href="https://colab.research.google.com/github/officiallong/Fork-Test-Run/blob/master/Exercise_2_ReACT_Prompting_Long_Nguyen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Exercise 2: Code Generation with ReACT Prompting**

**Student:** Long Nguyen  
**Tool:** Google Colab, Python, Google Gemini API  
**Goal:** Generate, test, observe, and improve Python code using a ReACT-style prompting workflow.

**## 1. Gemini API Setup**

In [6]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [7]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

print("Gemini client connected successfully.")

Gemini client connected successfully.


In [5]:
def ask_ai(prompt):
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    return response.text

In [9]:
react_prompt = """
You are an AI coding assistant using a ReACT-style workflow.

Your task is to create a Python function named analyze_orders()
that analyzes a list of customer order amounts.

Requirements:
- Input: a Python list containing customer order amounts.
- Valid order amounts must be numeric (int or float) and greater than or equal to 0.
- Ignore invalid values such as strings, None, and negative numbers.
- Return a dictionary containing:
    1. total_orders
    2. total_revenue
    3. average_order_value
    4. highest_order
    5. lowest_order
- Round monetary results to 2 decimal places.
- If there are no valid orders, return:
  {
      "total_orders": 0,
      "total_revenue": 0,
      "average_order_value": 0,
      "highest_order": None,
      "lowest_order": None
  }
- Use only standard Python. Do not use external libraries.

Follow these stages:

REASON/PLAN:
Briefly explain how you will solve the problem and handle invalid
or empty input.

GENERATE:
Write the complete Python function.

TEST:
Provide these exact test cases:

Test 1:
[25.50, 40, 15.75, 100, 60.25]

Test 2:
[25, "invalid", None, -10, 50.5]

Test 3:
[]

EXPECTED:
State what should happen for each test case.

Do not claim the code has been successfully executed.
I will execute it separately in Python and provide the observed result.
"""

react_response = ask_ai(react_prompt)

print(react_response)

**REASON/PLAN:**
1. **Input Validation & Filtering**: Iterate through the input list and filter out any invalid elements. A valid order must be an instance of `int` or `float` (specifically excluding `bool`, since `bool` is a subclass of `int` in Python) and must be greater than or equal to `0`.
2. **Handling Empty/No Valid Orders**: Check if the filtered list is empty. If so, return the specified default dictionary with counts and revenue set to `0`, and extremes set to `None`.
3. **Metric Calculations**:
   - `total_orders`: Compute the count using `len()`.
   - `total_revenue`: Compute the sum using `sum()`, rounded to 2 decimal places.
   - `average_order_value`: Divide `total_revenue` by `total_orders`, rounded to 2 decimal places.
   - `highest_order`: Find the maximum value using `max()`, rounded to 2 decimal places.
   - `lowest_order`: Find the minimum value using `min()`, rounded to 2 decimal places.
4. **Return Structure**: Construct and return the dictionary with all comput

In [10]:
def analyze_orders(orders):
    # Filter for valid numeric non-negative amounts, excluding booleans
    valid_orders = [
        val
        for val in orders
        if isinstance(val, (int, float))
        and not isinstance(val, bool)
        and val >= 0
    ]

    # Handle the empty/no valid orders case
    if not valid_orders:
        return {
            "total_orders": 0,
            "total_revenue": 0,
            "average_order_value": 0,
            "highest_order": None,
            "lowest_order": None,
        }

    total_orders = len(valid_orders)
    total_revenue = sum(valid_orders)
    average_order_value = total_revenue / total_orders
    highest_order = max(valid_orders)
    lowest_order = min(valid_orders)

    return {
        "total_orders": total_orders,
        "total_revenue": round(total_revenue, 2),
        "average_order_value": round(average_order_value, 2),
        "highest_order": round(highest_order, 2),
        "lowest_order": round(lowest_order, 2),
    }

In [11]:
# RUN - Execute the AI-generated code with the required test cases

test_1_input = [25.50, 40, 15.75, 100, 60.25]
test_2_input = [25, "invalid", None, -10, 50.5]
test_3_input = []

result_1 = analyze_orders(test_1_input)
result_2 = analyze_orders(test_2_input)
result_3 = analyze_orders(test_3_input)

print("TEST 1 RESULT:")
print(result_1)

print("\nTEST 2 RESULT:")
print(result_2)

print("\nTEST 3 RESULT:")
print(result_3)

TEST 1 RESULT:
{'total_orders': 5, 'total_revenue': 241.5, 'average_order_value': 48.3, 'highest_order': 100, 'lowest_order': 15.75}

TEST 2 RESULT:
{'total_orders': 2, 'total_revenue': 75.5, 'average_order_value': 37.75, 'highest_order': 50.5, 'lowest_order': 25}

TEST 3 RESULT:
{'total_orders': 0, 'total_revenue': 0, 'average_order_value': 0, 'highest_order': None, 'lowest_order': None}


In [13]:
observation_prompt = f"""
You are an AI coding assistant continuing a ReACT-style workflow.

The original Python function was tested with these results:

Test 1:
{result_1}

Test 2:
{result_2}

Test 3:
{result_3}

OBSERVE:
All three required tests passed successfully. However, the function
assumes that the input itself is a list. It should be more robust if
someone passes an invalid input such as None or a string.

FIX/IMPROVE:
Review the observed results and improve the function so that:
- It first checks whether the input is a Python list.
- If the input is not a list, return the same empty-results dictionary.
- Preserve all existing requirements and behavior.
- Use only standard Python.
- Keep the function name analyze_orders().
- Do not remove the existing handling for invalid individual values.

Return exactly these two sections:

OBSERVATION:
Briefly explain what worked and what limitation is being addressed.

IMPROVED CODE:
Provide the complete revised Python function only.
"""

improvement_response = ask_ai(observation_prompt)

print(improvement_response)

OBSERVATION:
The previous implementation correctly processed valid order lists and handled individual invalid items and empty lists. However, it lacked input validation for the `orders` parameter itself, causing potential exceptions (such as `TypeError`) if a non-list type (e.g., `None`, a string, or an integer) was passed. Adding an explicit `isinstance(orders, list)` check allows the function to safely return the standard empty-results dictionary when given invalid input types.

IMPROVED CODE:
```python
def analyze_orders(orders):
    empty_result = {
        'total_orders': 0,
        'total_revenue': 0,
        'average_order_value': 0,
        'highest_order': None,
        'lowest_order': None,
    }

    if not isinstance(orders, list):
        return empty_result

    valid_orders = [
        order
        for order in orders
        if isinstance(order, (int, float))
        and not isinstance(order, bool)
        and order > 0
    ]

    if not valid_orders:
        return em

In [14]:
fix_prompt = f"""
You are an AI coding assistant continuing a ReACT-style workflow.

Here is the revised code you previously generated:

{improvement_response}

OBSERVE:
The revised code successfully added validation for non-list inputs.
However, it introduced two regressions compared with the original
requirements:

1. Valid order amounts were originally defined as numbers greater than
   or equal to 0, but the revised code uses order > 0, which incorrectly
   excludes 0.
2. The original requirements specified that monetary results must be
   rounded to 2 decimal places, but the revised code no longer explicitly
   rounds those results.

FIX:
Revise the function while preserving the new input validation.

Requirements:
- Input must be a Python list. Otherwise, return the empty-results dictionary.
- Valid individual values must be int or float, excluding bool.
- Values greater than or equal to 0 are valid.
- Ignore strings, None, negative numbers, and booleans.
- Round total_revenue, average_order_value, highest_order, and lowest_order
  to 2 decimal places.
- Preserve the required empty-results dictionary.
- Use only standard Python.
- Keep the function name analyze_orders().

Return exactly these two sections:

FIX EXPLANATION:
Briefly explain what was corrected.

FINAL CODE:
Provide the complete corrected Python function.
"""

final_response = ask_ai(fix_prompt)

print(final_response)

FIX EXPLANATION:
1. Updated the filter condition from `order > 0` to `order >= 0` to ensure that zero-value orders are treated as valid amounts.
2. Applied `round(..., 2)` to `total_revenue`, `average_order_value`, `highest_order`, and `lowest_order` to meet the rounding requirement for monetary results.
3. Preserved input validation checking that `orders` is a `list`, as well as excluding `bool` types and invalid elements.

FINAL CODE:
```python
def analyze_orders(orders):
    empty_result = {
        'total_orders': 0,
        'total_revenue': 0,
        'average_order_value': 0,
        'highest_order': None,
        'lowest_order': None,
    }

    if not isinstance(orders, list):
        return empty_result

    valid_orders = [
        order
        for order in orders
        if isinstance(order, (int, float))
        and not isinstance(order, bool)
        and order >= 0
    ]

    if not valid_orders:
        return empty_result

    total_orders = len(valid_orders)
    total_

In [16]:
def analyze_orders(orders):
    empty_result = {
        'total_orders': 0,
        'total_revenue': 0,
        'average_order_value': 0,
        'highest_order': None,
        'lowest_order': None,
    }

    if not isinstance(orders, list):
        return empty_result

    valid_orders = [
        order
        for order in orders
        if isinstance(order, (int, float))
        and not isinstance(order, bool)
        and order >= 0
    ]

    if not valid_orders:
        return empty_result

    total_orders = len(valid_orders)
    total_revenue = sum(valid_orders)
    average_order_value = total_revenue / total_orders

    return {
        'total_orders': total_orders,
        'total_revenue': round(total_revenue, 2),
        'average_order_value': round(average_order_value, 2),
        'highest_order': round(max(valid_orders), 2),
        'lowest_order': round(min(valid_orders), 2),
    }

In [18]:
# FINAL RUN - Test the corrected function

final_tests = {
    "Test 1 - Valid orders": [25.50, 40, 15.75, 100, 60.25],
    "Test 2 - Mixed valid/invalid": [25, "invalid", None, -10, 50.5],
    "Test 3 - Empty list": [],
    "Test 4 - Zero value": [0, 25.75, 50],
    "Test 5 - Boolean value": [True, False, 20, 30],
    "Test 6 - None input": None,
    "Test 7 - String input": "invalid"
}

for test_name, test_input in final_tests.items():
    print(test_name)
    print("Input:", test_input)
    print("Output:", analyze_orders(test_input))
    print()

Test 1 - Valid orders
Input: [25.5, 40, 15.75, 100, 60.25]
Output: {'total_orders': 5, 'total_revenue': 241.5, 'average_order_value': 48.3, 'highest_order': 100, 'lowest_order': 15.75}

Test 2 - Mixed valid/invalid
Input: [25, 'invalid', None, -10, 50.5]
Output: {'total_orders': 2, 'total_revenue': 75.5, 'average_order_value': 37.75, 'highest_order': 50.5, 'lowest_order': 25}

Test 3 - Empty list
Input: []
Output: {'total_orders': 0, 'total_revenue': 0, 'average_order_value': 0, 'highest_order': None, 'lowest_order': None}

Test 4 - Zero value
Input: [0, 25.75, 50]
Output: {'total_orders': 3, 'total_revenue': 75.75, 'average_order_value': 25.25, 'highest_order': 50, 'lowest_order': 0}

Test 5 - Boolean value
Input: [True, False, 20, 30]
Output: {'total_orders': 2, 'total_revenue': 50, 'average_order_value': 25.0, 'highest_order': 30, 'lowest_order': 20}

Test 6 - None input
Input: None
Output: {'total_orders': 0, 'total_revenue': 0, 'average_order_value': 0, 'highest_order': None, 'low

# **Testing and Iteration:**

I first tested the AI-generated function using three test cases: a list of valid orders, a list containing both valid and invalid values, and an empty list. All three tests produced the expected results.

After reviewing the code, I noticed that it assumed the input itself would always be a list. I asked the AI to improve the function so that invalid inputs, such as None or a string, would not cause an error. The revised version added input validation, but it also changed the original condition from >= 0 to > 0 and removed the two-decimal rounding requirement.

I provided those observations back to the AI and asked it to correct the issues while keeping the new input validation. The final version restored zero as a valid order amount, added the required rounding, and preserved the new validation. I then ran seven test cases covering valid orders, mixed values, an empty list, zero values, booleans, None, and a string. All seven tests produced the expected results.